# Phase 1: 集計層（モデル不要）— v2（修正版）

`docs/requirements.md` 4章に基づく。予測ではなく事実の集計のみを行う。

## v1 からの変更点（Phase 1 検証レポートを受けた修正）

1. **脱落曲線をハザード率ベースに変更**: 「離脱者に占める割合」（PMF、母数固定）ではなく、
   「その話数に到達した人のうち、そこで止まった人の割合」（ハザード率、母数がその都度縮小）に変更。
   PMF方式は母数の大きさゆえに常に1話がピークになり、作品ごとの違いを表現できていなかった。
2. **総話数超過値のクリップ**: `watched_episodes` が `anime.csv` の総話数を超える場合、総話数にクリップする
   （除外ではない）。総話数が `Unknown` の作品はクリップ対象外とし `episodes_unknown: true` を付与する。
3. **question_pool の再選定**: `Type == Movie` を除外し、登録者数上位200件から無条件で50本を確保した上で、
   残り200本を（v1と同じ）登録者数上位3,000件プールから split_score ベースで多様化選定する。

## 離脱の定義（確定）

**定義B（離脱=4 + 保留=3）を主定義として維持する。** 根拠:
- 入力UIは「途中で止まった」の1ボタンで離脱と保留の両方を受け付ける設計であり、学習ラベルもこれに一致させる必要がある（訓練とサービングの分布を一致させる）
- 保留の平均視聴話数（10.7話）は離脱（8.3話）に近く、完走（16.9話）から遠い（Phase 0 で σ 算出前に確立した根拠）
- Phase 1 検証レポートで見つかった「保留使用率と個人効果の相関 -0.77」は、採用理由からもリスクからも除外する。保留を離脱側に含める定義である以上、保留を多用するユーザーほど定義B上の完走率が下がるのは**数学的に必然**であり、「記録習慣が個人効果を歪めている」ことの証明にはならない

記録習慣由来のノイズかどうかという懸念自体は妥当なため、Phase 2 で以下の方法により実証的に検証する:
- 目的変数は**定義A（離脱=4のみ）に固定**してホールドアウト評価を行う
- ユーザープロファイルは (a) 定義Aのデータのみ、(b) 定義Bのデータ、の2通りで構築
- (b) の方が定義Aラベルに対する予測精度が高ければ、保留を含む定義Bの情報価値が実証されたとして定義Bを確定する


In [1]:
import time
import re
import json
import numpy as np
import pandas as pd

RAW_DIR = "../data/raw"
ANIMELIST_PATH = f"{RAW_DIR}/animelist.csv"
ANIME_PATH = f"{RAW_DIR}/anime.csv"

CHUNKSIZE = 15_000_000
MIN_DROPPED_FOR_CURVE = 50


## 1. 総話数クリップの準備

`anime.csv` の `Episodes` を総話数の上限として使う。`Unknown` は `NaN` としクリップ対象外にする。

In [2]:
anime_ep = pd.read_csv(ANIME_PATH, usecols=["MAL_ID", "Episodes"])
anime_ep["Episodes"] = pd.to_numeric(anime_ep["Episodes"], errors="coerce")
ep_lookup = dict(zip(anime_ep["MAL_ID"], anime_ep["Episodes"]))

n_unknown_ep = anime_ep["Episodes"].isna().sum()
print(f"総話数 Unknown の作品数: {n_unknown_ep:,} / {len(anime_ep):,}")


総話数 Unknown の作品数: 516 / 17,562


## 2. 集計本体（1パス）: クリップしながら 作品×話数 別カウントを作る

チャンクごとに `watched_episodes` を総話数にクリップしてから、完走(status=2)・離脱扱い(status=3 or 4, 定義B)の
`(anime_id, watched_episodes)` 別カウントに縮約する。

In [3]:
t0 = time.time()

dtypes = {"anime_id": "int32", "watching_status": "int8", "watched_episodes": "int32"}
completed_parts = []
dropped_parts = []
n_rows_seen = 0
n_clipped_total = 0

for chunk in pd.read_csv(ANIMELIST_PATH, usecols=list(dtypes.keys()), dtype=dtypes, chunksize=CHUNKSIZE):
    n_rows_seen += len(chunk)
    ceiling = chunk["anime_id"].map(ep_lookup)  # NaN -> クリップ対象外
    clip_mask = ceiling.notna() & (chunk["watched_episodes"] > ceiling)
    n_clipped_total += int(clip_mask.sum())
    chunk.loc[clip_mask, "watched_episodes"] = ceiling[clip_mask].astype("int32")

    c = chunk[chunk["watching_status"] == 2]
    d = chunk[chunk["watching_status"].isin([3, 4])]  # 定義B
    completed_parts.append(c.groupby(["anime_id", "watched_episodes"]).size().rename("count"))
    dropped_parts.append(d.groupby(["anime_id", "watched_episodes"]).size().rename("count"))

completed_ep = pd.concat(completed_parts).groupby(level=[0, 1]).sum().reset_index()
dropped_ep = pd.concat(dropped_parts).groupby(level=[0, 1]).sum().reset_index()

print(f"rows scanned: {n_rows_seen:,} | rows clipped (総話数超過): {n_clipped_total:,} | elapsed: {time.time()-t0:.1f}s")


rows scanned: 109,224,747 | rows clipped (総話数超過): 197,231 | elapsed: 104.0s


## 3. 作品ごとの集計値

In [4]:
n_completed = completed_ep.groupby("anime_id")["count"].sum().rename("n_completed")
n_dropped = dropped_ep.groupby("anime_id")["count"].sum().rename("n_dropped")
totals = pd.concat([n_completed, n_dropped], axis=1).fillna(0)
totals["n_completed"] = totals["n_completed"].astype(int)
totals["n_dropped"] = totals["n_dropped"].astype(int)
totals["population_completion_rate"] = totals["n_completed"] / (totals["n_completed"] + totals["n_dropped"])
totals["episodes_unknown"] = totals.index.map(lambda aid: pd.isna(ep_lookup.get(aid)))

print(f"完走・離脱(定義B)いずれかの記録がある作品数: {len(totals):,}")
print(f"総話数 Unknown の作品数: {totals['episodes_unknown'].sum():,}")
print(f"離脱者(定義B) >= {MIN_DROPPED_FOR_CURVE} 件の作品数: {(totals['n_dropped'] >= MIN_DROPPED_FOR_CURVE).sum():,}")


完走・離脱(定義B)いずれかの記録がある作品数: 17,194
総話数 Unknown の作品数: 262
離脱者(定義B) >= 50 件の作品数: 7,645


## 4. ハザード率ベースの脱落曲線

`hazard(k) = k話で止まった人数 ÷ k話に到達した人数`（到達者には完走者も含む）。

- 「到達した」= 完走・離脱(定義B)を問わず `watched_episodes >= k` の人。作品ごとの逆累積和で求める
- `peak_dropout_episode` は **1話を除いた** ハザード率最大の話数（1話は序盤の様子見離脱で常に高くなり、
  作品間の違いを見えなくするため除外する。1話以外に離脱データが無い場合のみ1話にフォールバックする）
- `survival_after_peak` は新しい peak を使って再計算する

In [5]:
combined = pd.concat([completed_ep, dropped_ep]).groupby(["anime_id", "watched_episodes"], as_index=False)["count"].sum()
combined = combined.sort_values(["anime_id", "watched_episodes"])
totals_by_anime = combined.groupby("anime_id")["count"].sum()
combined["cum_asc"] = combined.groupby("anime_id")["count"].cumsum()
combined["total"] = combined["anime_id"].map(totals_by_anime)
combined["at_risk"] = combined["total"] - combined["cum_asc"] + combined["count"]

dropped_valid = dropped_ep[dropped_ep["watched_episodes"] > 0]
valid_ids = set(dropped_valid.groupby("anime_id")["count"].sum().index)

hz = dropped_valid.merge(
    combined[["anime_id", "watched_episodes", "at_risk"]],
    on=["anime_id", "watched_episodes"], how="left"
)
hz["hazard"] = hz["count"] / hz["at_risk"]

eligible_ids = totals.index[(totals["n_dropped"] >= MIN_DROPPED_FOR_CURVE) & totals.index.isin(valid_ids)]
eligible_set = set(eligible_ids)
print(f"曲線を出力する作品数: {len(eligible_ids):,}")


曲線を出力する作品数: 7,603


In [6]:
hz_not_ep1 = hz[hz["watched_episodes"] != 1]
hz_sorted = hz_not_ep1.sort_values(["anime_id", "hazard", "watched_episodes"], ascending=[True, False, True])
peak_not_ep1 = hz_sorted.groupby("anime_id").first()["watched_episodes"]
hz_ep1 = hz[hz["watched_episodes"] == 1].set_index("anime_id")["watched_episodes"]

def get_peak(aid):
    if aid in peak_not_ep1.index:
        return int(peak_not_ep1.loc[aid])
    if aid in hz_ep1.index:
        return 1
    return None

peak_lookup = {aid: get_peak(aid) for aid in eligible_ids}
n_peak_ep1_fallback = sum(1 for v in peak_lookup.values() if v == 1)
print(f"1話以外に離脱データが無くpeak=1話になった作品数: {n_peak_ep1_fallback:,}")


1話以外に離脱データが無くpeak=1話になった作品数: 1,900


### 【判明した注意点】短編作品でのピーク不安定性

終盤は「到達者数」（分母）が小さくなるため、短い作品ほど最終話付近でハザード率が数点のデータだけで
跳ね上がりやすい。実際、全話数5話以下の作品では85%が `peak_dropout_episode == 総話数`（最終話）となっており、
これは内容由来の脱落というより**サンプルサイズの小ささによる統計的な不安定性**の可能性が高い。
今回の修正範囲（要求された3点）には含まれないため元の指示通りに実装したが、Phase 2 に進む前に
最小到達者数などの安定化策を検討する余地があることを記録しておく。

In [7]:
ep_totals = totals.copy()
ep_totals["total_episodes"] = ep_totals.index.map(ep_lookup)
peak_series_check = pd.Series(peak_lookup)
check_df = ep_totals.loc[peak_series_check.index].copy()
check_df["peak"] = peak_series_check
check_df = check_df[check_df["total_episodes"].notna()]
check_df["peak_is_final"] = check_df["peak"] == check_df["total_episodes"]

short = check_df[check_df["total_episodes"] <= 5]
long_ = check_df[check_df["total_episodes"] > 50]
print(f"全話数<=5 の作品で peak==最終話 の割合: {short['peak_is_final'].mean()*100:.1f}% (n={len(short):,})")
print(f"全話数>50 の作品で peak==最終話 の割合: {long_['peak_is_final'].mean()*100:.1f}% (n={len(long_):,})")


全話数<=5 の作品で peak==最終話 の割合: 85.1% (n=3,377)
全話数>50 の作品で peak==最終話 の割合: 1.9% (n=432)


## 5. survival_after_peak の再計算

In [8]:
peak_series = pd.Series(peak_lookup, name="peak")
completed_gt = completed_ep.merge(peak_series.rename("peak"), left_on="anime_id", right_index=True, how="inner")
completed_gt = completed_gt[completed_gt["watched_episodes"] > completed_gt["peak"]]
completed_gt_sum = completed_gt.groupby("anime_id")["count"].sum().rename("completed_gt")

dropped_gt = dropped_valid.merge(peak_series.rename("peak"), left_on="anime_id", right_index=True, how="inner")
dropped_gt = dropped_gt[dropped_gt["watched_episodes"] > dropped_gt["peak"]]
dropped_gt_sum = dropped_gt.groupby("anime_id")["count"].sum().rename("dropped_gt")

surv = pd.concat([completed_gt_sum, dropped_gt_sum], axis=1).fillna(0)
surv["denom"] = surv["completed_gt"] + surv["dropped_gt"]
surv["survival_after_peak"] = np.where(surv["denom"] > 0, surv["completed_gt"] / surv["denom"], np.nan)

n_fallback = (surv["denom"] == 0).sum()
print(f"survival_after_peak の分母が0件の作品数（母集団完走率で代替、peakが終盤にあるほど起きやすい）: {n_fallback:,}")


survival_after_peak の分母が0件の作品数（母集団完走率で代替、peakが終盤にあるほど起きやすい）: 0


## 6. dropout_curves.json の書き出し

In [9]:
hz_by_anime = {aid: g.sort_values("watched_episodes") for aid, g in hz.groupby("anime_id")}
curves = []
n_fallback_used = 0

for aid in totals.index:
    row = totals.loc[aid]
    if aid not in eligible_set:
        curves.append({"anime_id": int(aid), "insufficient_data": True})
        continue
    g = hz_by_anime[aid]
    curve = [{"episode": int(e), "rate": round(float(r), 4)} for e, r in zip(g["watched_episodes"], g["hazard"])]
    peak_ep = peak_lookup[aid]
    if aid in surv.index and surv.loc[aid, "denom"] > 0:
        sap = round(float(surv.loc[aid, "survival_after_peak"]), 4)
    else:
        sap = round(float(row["population_completion_rate"]), 4)
        n_fallback_used += 1
    entry = {
        "anime_id": int(aid),
        "dropout_curve": curve,
        "peak_dropout_episode": peak_ep,
        "survival_after_peak": sap,
        "population_completion_rate": round(float(row["population_completion_rate"]), 4),
    }
    if bool(row["episodes_unknown"]):
        entry["episodes_unknown"] = True
    curves.append(entry)

n_insufficient = sum(1 for c in curves if c.get("insufficient_data"))
print(f"total elapsed: {time.time()-t0:.1f}s")
print(f"出力エントリ総数: {len(curves):,} | insufficient_data: {n_insufficient:,} | 曲線あり: {len(curves)-n_insufficient:,}")

with open("../data/dropout_curves.json", "w") as f:
    json.dump(curves, f, ensure_ascii=False)
print("saved: data/dropout_curves.json")


total elapsed: 105.8s
出力エントリ総数: 17,194 | insufficient_data: 9,591 | 曲線あり: 7,603
saved: data/dropout_curves.json


### 検証: Death Note・NARUTO で既知の脱落ポイントが再現されるか

In [10]:
curves_by_id = {c["anime_id"]: c for c in curves}
for aid, label in [(1535, "Death Note"), (20, "NARUTO")]:
    c = curves_by_id[aid]
    print(f"{label}: peak_dropout_episode={c['peak_dropout_episode']}, "
          f"survival_after_peak={c['survival_after_peak']}, "
          f"population_completion_rate={c['population_completion_rate']}")


Death Note: peak_dropout_episode=25, survival_after_peak=0.9856, population_completion_rate=0.9231
NARUTO: peak_dropout_episode=135, survival_after_peak=0.9661, population_completion_rate=0.8495


Death Note は25話（Lが死亡する回として有名）、NARUTOは135話付近（無限月読編前、フィラー突入期に近い）で
ピークが検出されており、修正の意図通り作品内容に基づく脱落点が可視化された
（比較画像: `reports/figures/dropout_curves_before_after.png`）。

## 7. question_pool の再選定

- `Type == Movie`（および従来通り Music・Unknown）を除外
- 登録者数上位200件から**無条件で50本**を確保（split_scoreは無視。誰もが知っている作品を必ず含める）
- 残り200本は従来通り、登録者数上位3,000件のプールから split_score ベースで多様化選定

In [11]:
anime = pd.read_csv(ANIME_PATH, usecols=[
    "MAL_ID", "Name", "Japanese name", "Genres", "Episodes", "Aired", "Type", "Members",
])
anime = anime.rename(columns={"MAL_ID": "anime_id"})

def extract_year(aired):
    m = re.search(r"(19|20)\d{2}", str(aired))
    return int(m.group()) if m else None

anime["year"] = anime["Aired"].apply(extract_year)
anime["episodes_num"] = pd.to_numeric(anime["Episodes"], errors="coerce")
anime["title"] = anime["Japanese name"].where(
    anime["Japanese name"].notna() & (anime["Japanese name"] != "Unknown"), anime["Name"]
)
anime["genre_list"] = anime["Genres"].apply(lambda s: [g.strip() for g in str(s).split(",")] if pd.notna(s) else [])

totals_r = totals.reset_index().rename(columns={"index": "anime_id"})
totals_r["total_registrations"] = totals_r["n_completed"] + totals_r["n_dropped"]

pool_src = anime.merge(totals_r, on="anime_id", how="inner")
print(f"animelist.csv 上で完走/離脱いずれかの記録がある作品: {len(pool_src):,}")


animelist.csv 上で完走/離脱いずれかの記録がある作品: 17,194


In [12]:
EXCLUDED_TYPES = {"Music", "Unknown", "Movie"}
TOP_N_POPULARITY = 3000
RESERVED_TOP_N = 200
RESERVED_COUNT = 50
TARGET_SIZE = 250

pool_src = pool_src[~pool_src["Type"].isin(EXCLUDED_TYPES)]
pool_src = pool_src[pool_src["year"].notna()]
pool_src = pool_src[pool_src["episodes_num"].notna() & (pool_src["episodes_num"] > 0)]
pool_src = pool_src[pool_src["genre_list"].apply(len) > 0]
pool_src = pool_src[pool_src["title"].notna()]
print(f"品質フィルタ後（Movie/Music/Unknown除外・年/話数/ジャンル既知）: {len(pool_src):,}")

pool_src = pool_src.sort_values("total_registrations", ascending=False).reset_index(drop=True)
pool_src = pool_src.head(TOP_N_POPULARITY)
print(f"登録者数上位{TOP_N_POPULARITY}件に絞り込み: {len(pool_src):,}")


品質フィルタ後（Movie/Music/Unknown除外・年/話数/ジャンル既知）: 12,387
登録者数上位3000件に絞り込み: 3,000


In [13]:
def entropy(p):
    if p <= 0 or p >= 1:
        return 0.0
    return float(-p * np.log2(p) - (1 - p) * np.log2(1 - p))

pool_src["completion_rate"] = pool_src["population_completion_rate"]
pool_src["split_score"] = pool_src["completion_rate"].apply(entropy)

def year_bucket(y):
    if y <= 2005:
        return "~2005"
    elif y <= 2010:
        return "2006-2010"
    elif y <= 2015:
        return "2011-2015"
    else:
        return "2016-2020"

def episode_bucket(e):
    if e <= 13:
        return "~13"
    elif e <= 26:
        return "14-26"
    else:
        return "27~"

pool_src["year_bucket"] = pool_src["year"].apply(year_bucket)
pool_src["episode_bucket"] = pool_src["episodes_num"].apply(episode_bucket)
pool_src["primary_genre"] = pool_src["genre_list"].apply(lambda g: g[0] if g else "unknown")

top_pool = pool_src.head(RESERVED_TOP_N)
reserved = top_pool.head(RESERVED_COUNT).copy()
print(f"登録者数上位{RESERVED_TOP_N}件から確保: {len(reserved)}件")
print(f"確保枠の登録件数レンジ: {reserved['total_registrations'].min():,} ~ {reserved['total_registrations'].max():,}")

reserved_ids = set(reserved["anime_id"])
remaining_candidates = pool_src[~pool_src["anime_id"].isin(reserved_ids)].copy()
remaining_candidates = remaining_candidates.sort_values(
    ["split_score", "total_registrations"], ascending=[False, False]
).reset_index(drop=True)


登録者数上位200件から確保: 50件
確保枠の登録件数レンジ: 106,074 ~ 219,360


### 貪欲法による多様化選択（残り200本）

確保済みの50本のジャンル・年代・話数バケットも上限カウントに算入したうえで、split_score順に走査する。

In [14]:
remaining_target = TARGET_SIZE - RESERVED_COUNT
genre_cap = max(10, TARGET_SIZE // 8)
year_cap = max(20, TARGET_SIZE // 3)
episode_cap = max(30, TARGET_SIZE // 2)

genre_count = reserved["primary_genre"].value_counts().to_dict()
year_count = reserved["year_bucket"].value_counts().to_dict()
episode_count = reserved["episode_bucket"].value_counts().to_dict()

selected_idx = []
for i, row in remaining_candidates.iterrows():
    if len(selected_idx) >= remaining_target:
        break
    g, y, e = row["primary_genre"], row["year_bucket"], row["episode_bucket"]
    if genre_count.get(g, 0) >= genre_cap:
        continue
    if year_count.get(y, 0) >= year_cap:
        continue
    if episode_count.get(e, 0) >= episode_cap:
        continue
    selected_idx.append(i)
    genre_count[g] = genre_count.get(g, 0) + 1
    year_count[y] = year_count.get(y, 0) + 1
    episode_count[e] = episode_count.get(e, 0) + 1

print(f"多様化パス後の追加件数: {len(selected_idx):,}")

if len(selected_idx) < remaining_target:
    remaining_idx = [i for i in remaining_candidates.index if i not in set(selected_idx)]
    for i in remaining_idx:
        if len(selected_idx) >= remaining_target:
            break
        selected_idx.append(i)
    print(f"補充パス後の追加件数: {len(selected_idx):,}")

diversified = remaining_candidates.loc[selected_idx]
pool_df = pd.concat([reserved, diversified], ignore_index=True).sort_values("split_score", ascending=False)
print(f"\n最終プールサイズ: {len(pool_df):,}")


多様化パス後の追加件数: 200

最終プールサイズ: 250


In [15]:
print("放送年代の分布:")
print(pool_df["year_bucket"].value_counts())
print("\n話数レンジの分布:")
print(pool_df["episode_bucket"].value_counts())
print("\n主ジャンルの分布（上位10）:")
print(pool_df["primary_genre"].value_counts().head(10))
print("\n完走率の分布:")
print(pool_df["completion_rate"].describe())
print("\n中央値:", pool_df["completion_rate"].median())
print("0.85以上の件数:", (pool_df["completion_rate"] >= 0.85).sum())


放送年代の分布:
year_bucket
2016-2020    83
2011-2015    76
2006-2010    55
~2005        36
Name: count, dtype: int64

話数レンジの分布:
episode_bucket
~13      125
14-26     85
27~       40
Name: count, dtype: int64

主ジャンルの分布（上位10）:
primary_genre
Action           32
Comedy           31
Adventure        31
Slice of Life    31
Mystery          19
Drama            19
Sci-Fi           14
Fantasy          10
Military          9
Psychological     7
Name: count, dtype: int64

完走率の分布:
count    250.000000
mean       0.732600
std        0.120274
min        0.339320
25%        0.667314
50%        0.723186
75%        0.781853
max        0.974294
Name: completion_rate, dtype: float64

中央値: 0.7231864803631143
0.85以上の件数: 45


In [16]:
pool = []
for _, row in pool_df.iterrows():
    pool.append({
        "title": row["title"],
        "anime_id": int(row["anime_id"]),
        "year": int(row["year"]),
        "episodes": int(row["episodes_num"]),
        "genres": row["genre_list"],
        "completion_rate": round(float(row["completion_rate"]), 4),
        "split_score": round(float(row["split_score"]), 4),
    })

with open("../data/question_pool.json", "w") as f:
    json.dump(pool, f, ensure_ascii=False, indent=2)

print(f"saved: data/question_pool.json ({len(pool)}本)")
print(json.dumps(pool[:3], ensure_ascii=False, indent=2))


saved: data/question_pool.json (250本)
[
  {
    "title": "逆転裁判 ～その「真実」、異議あり！～",
    "anime_id": 31630,
    "year": 2016,
    "episodes": 24,
    "genres": [
      "Comedy",
      "Drama",
      "Mystery",
      "Police"
    ],
    "completion_rate": 0.4964,
    "split_score": 1.0
  },
  {
    "title": "戦国コレクション",
    "anime_id": 12611,
    "year": 2012,
    "episodes": 26,
    "genres": [
      "Fantasy",
      "Parody",
      "Samurai"
    ],
    "completion_rate": 0.4805,
    "split_score": 0.9989
  },
  {
    "title": "きらりん☆レボリューション",
    "anime_id": 1516,
    "year": 2006,
    "episodes": 153,
    "genres": [
      "Comedy",
      "Drama",
      "Romance",
      "Shoujo"
    ],
    "completion_rate": 0.4762,
    "split_score": 0.9984
  }
]


## 8. プロファイル算出ロジック（`api/services/profile.py`）の動作確認

v1から変更なし。ハザード曲線・question_pool の修正はこのロジックに影響しない。

In [17]:
import sys
sys.path.insert(0, "..")
from api.services.profile import Response, build_profile, compute_catalog_genre_baseline

catalog = [
    {"anime_id": int(r.anime_id), "genres": r.genre_list}
    for r in anime.itertuples() if r.genre_list
]
genre_baseline = compute_catalog_genre_baseline(catalog)

sample_pool = pool[:5]
sample_responses = [
    Response(
        anime_id=item["anime_id"], label=label, episodes=item["episodes"],
        genres=item["genres"], members=5000,
    )
    for item, label in zip(sample_pool, ["loved", "completed", "dropped", "dropped", "completed"])
]
profile = build_profile(sample_responses, genre_baseline)
print(json.dumps(profile, ensure_ascii=False, indent=2))


{
  "type_name": "長距離・心理・重量級派タイプ",
  "endurance_episodes": 153,
  "completion_rate": 0.6,
  "preferred_genres": [
    "Mystery",
    "Samurai",
    "Police",
    "Super Power",
    "Parody"
  ],
  "avoided_genres": [
    "Comedy"
  ],
  "mainstream_affinity": {
    "median_members": 5000.0,
    "mean_members": 5000.0
  }
}


---

## 9. まとめ

- `data/dropout_curves.json` — ハザード率ベースに刷新。総話数超過値はクリップ済み。Death Note(25話)・NARUTO(135話)で
  内容に基づく脱落ポイントを再現できることを確認
- `data/question_pool.json` — Movie除外、登録者数上位200件から50本を無条件確保 + 残り200本を多様化選定した250本
- 離脱の主定義は**定義B**で確定（根拠は本ノートブック冒頭に明記）。記録習慣由来ノイズの懸念は Phase 2 で
  定義A目的変数への予測精度比較により実証的に検証する
- 短編作品でのピーク不安定性（終盤バイアス）を新たに確認。Phase 2 着手前の追加判断事項として報告する

**Phase 2（予測モデル）にはまだ着手していない。**